# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rattan-Kumar/flyrank-ML-Intership/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    average_precision_score,
    roc_auc_score
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("intership")

con = duckdb.connect()

rel = "hf://datasets/FlyRank/internship-warehouse"

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_token
    (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')
    """
)

print("FlyRank warehouse connection successful.")

FlyRank warehouse connection successful.


In [ ]:
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_data_available,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available = TRUE
"""

df = con.sql(query).df()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

For Week 5, I will use **Logistic Regression** as the machine learning method. It fits this problem because the Week 4 baseline is a binary decision: a content item is labeled **`CTR_FIX_CANDIDATE`** if it has at least 100 impressions and a CTR below 2%; otherwise, it is labeled **`NO_ACTION`**.

Logistic Regression is a simple and easy-to-understand model that can learn how different search-related signals are connected to these decisions. It also gives a probability score, which can be used to rank content items based on how likely they are to be CTR improvement candidates.

I chose to start with a simple model instead of jumping directly to a complex one. The main goal this week is to see whether machine learning can provide useful decision support beyond the transparent Week 4 rule. A more complex model is not automatically better just because it is more advanced.

One important limitation is that the Week 4 label comes directly from the baseline rule, rather than from an independent future outcome. This means the experiment mainly tests how well the ML model can reproduce the existing rule. It does not prove that the pages selected by the model will actually benefit from a CTR improvement.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Calculate CTR (Click-Through Rate)
df["ctr_pct"] = (df["gsc_clicks"] / df["gsc_impressions"]) * 100

# Create the binary target from the Week-4 baseline decision.
# A row is marked as CTR_FIX_CANDIDATE (target=1) when it has at least 100 impressions and CTR below 2%.
# Otherwise, it receives NO_ACTION (target=0).
df["target"] = ((df["gsc_impressions"] >= 100) & (df["ctr_pct"] < 2)).astype(int)

# Use only decision-time numeric signals.
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr_pct"
]

X = df[feature_columns].copy()
y = df["target"].copy()

# Create a reproducible 80/20 stratified split.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.